# MLP victim: FEVI vs. BDQ and TRPO

The victim is a one-hidden-layer network `f(x; θ) = W₂ tanh(W₁x + b₁) + b₂`, trained by online
SGD on the squared loss, where `θ` is the flattened parameter vector (D = 57 at d = 5 with 8
hidden units). The data follow `y = sin(⟨θ*, x⟩) + noise`. One SGD step on the reweighted
batch is still `θ + Σ aᵢ φᵢ`, since the batch gradient is a sum over points; the check below
confirms it. FEVI therefore applies, with `θ` now in parameter space.

- **BDQ**: branching Q-network, one 3-way head per point (Tavakoli et al., 2018).
- **TRPO**: actor-critic over delete/duplicate swaps with a KL trust region.

All methods share the environment, feasibility constraints, seeds and evaluation episodes.
Cost is the attacker's episode cost, so lower is better.

In [ ]:
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import time
from dataclasses import replace

import numpy as np
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from config import EnvConfig, MethodConfig
from env import Environment, is_feasible
from fvi import train_fevi, evaluate, StagedValue
from agents import NullAgent, GreedyAgent, MethodAgent
from bdq import train_bdq, BDQAgent, BranchQNet
from trpo import train_trpo, TRPOAgent, SwapPolicy, StateValue, max_swaps

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "font.size": 11, "legend.frameon": False,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": "0.85",
})

## Setup

`QUICK = True` runs 1 seed for 20 iterations. Every learner trains for `K_COMPARE`
iterations of `N_episodes` episodes each.

`eta = 0.3`: the victim's own SGD diverges above about 0.5. `φᵢ` needs per-sample gradients,
so every step is slower than in the linear notebook. The edit price `c_edit = 0.05`
(λ = 2m·c_edit = 1) comes from a small sweep.

In [ ]:
QUICK = False

ECFG = EnvConfig(d=5, m=10, T=10, eta=0.3, noise=0.1, task="mlp",
                 victim_hidden=8, init_scale=0.3)
CFG = MethodConfig(c_edit=0.05, weight_decay=1e-3)

N_SEEDS = 1 if QUICK else 10          # number of seeds to run
SEEDS = tuple(range(N_SEEDS))        # 0, 1, ..., N_SEEDS - 1
K_COMPARE = 20 if QUICK else 50
EVAL_EPISODES, EVAL_SEED = 200, 999

FEVI_M = 5
FEVI_N = None

OURS = "FEVI"
LEARNERS = [OURS, "BDQ", "TRPO"]
NAMES = ["null", "greedy", "TRPO", "BDQ", OURS]
COLORS = {"null": "#5b5b5b", "greedy": "#eb6834", "TRPO": "#4a3aa7",
          "BDQ": "#e34948", OURS: "#2a78d6"}
MARKERS = {"null": "D", "greedy": "s", "TRPO": "^", "BDQ": "X", OURS: "o"}


def fevi_cfg(cfg, T):
    return replace(cfg, select_mode="greedy", fevi_outer=cfg.K_iters,
                   fevi_N=FEVI_N or cfg.N_episodes, fevi_M=FEVI_M,
                   fevi_fit_steps=max(1, cfg.n_grad_steps // T))


def train_ours(env, cfg, seed, **kw):
    return train_fevi(env, fevi_cfg(cfg, env.T), seed=seed, verbose=False, **kw)


def print_raw(title, rows, digits=3):
    n = len(next(iter(rows.values())))
    head = f"{title:26}{'mean':>10}{'std':>9}"
    if n > 1:
        head += "".join(f"{f'seed {s}':>10}" for s in SEEDS[:n])
    print("\n" + head)
    print("-" * len(head))
    for name, v in rows.items():
        v = np.asarray(v, dtype=float)
        line = f"{name:26}{v.mean():10.{digits}f}{v.std():9.{digits}f}"
        if n > 1:
            line += "".join(f"{x:10.{digits}f}" for x in v)
        print(line)

### Check: the update is linear in the weights

In [ ]:
env0 = Environment(ECFG, CFG)
z = env0.stream.sample_batch(env0.m, torch.Generator().manual_seed(1))
a = torch.tensor([0., 2.] * (env0.m // 2) + [1.] * (env0.m % 2))     # a feasible reweighting
th = env0.theta_0.clone()

linear = th + a @ env0.victim.phi_batch(th, z)
thg = th.clone().requires_grad_(True)
loss = (a * 0.5 * (env0.victim.forward(thg, z[0]) - z[1]) ** 2).sum() / env0.m
direct = th - ECFG.eta * torch.autograd.grad(loss, thg)[0]              # explicit SGD step

err = float((linear - direct).abs().max())
print(f"D = {env0.d} parameters;  max |θ + aΦ − explicit SGD step| = {err:.2e}")
assert err < 1e-5

### Model size and training budget

In [ ]:
def count_params(net):
    return sum(p.numel() for p in net.parameters())

_env = Environment(ECFG, CFG)
_fe = fevi_cfg(replace(CFG, K_iters=K_COMPARE), _env.T)
params = {
    OURS: count_params(StagedValue(_env.d, _env.T, CFG.hidden)),
    "BDQ": count_params(BranchQNet(_env.d, _env.T, CFG.hidden)),
    "TRPO": count_params(SwapPolicy(_env.d, _env.T, max_swaps(_env.m, CFG.B), CFG.hidden))
            + count_params(StateValue(_env.d, _env.T, CFG.hidden)),
}
episodes = {OURS: _fe.fevi_outer * _fe.fevi_N, "BDQ": K_COMPARE * CFG.N_episodes,
            "TRPO": K_COMPARE * CFG.N_episodes}
grad_steps = {OURS: _fe.fevi_outer * _env.T * _fe.fevi_fit_steps,
              "BDQ": K_COMPARE * CFG.n_grad_steps, "TRPO": None}
draws = {OURS: _fe.fevi_outer * _env.T * _fe.fevi_N * _fe.fevi_M, "BDQ": 0, "TRPO": 0}

print(f"{'':24}" + "".join(f"{n:>12}" for n in LEARNERS))
for label, row in [("parameters", params), ("training episodes", episodes),
                   ("gradient steps", grad_steps), ("extra batches from P", draws)]:
    print(f"{label:24}" + "".join(f"{row[n]:>12,}" if row[n] is not None else f"{'-':>12}"
                                  for n in LEARNERS))

## Training

Each seed trains all three learners from the same initialisation, then scores every policy
on the same evaluation episodes.

In [ ]:
def run_seed(s, K):
    cfg_k = replace(CFG, K_iters=K)
    torch.manual_seed(s)
    env = Environment(ECFG, cfg_k)
    eval_kw = dict(return_history=True, best_checkpoint=True,
                   eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
    hist, times = {}, {}

    torch.manual_seed(s)
    t0 = time.perf_counter()
    V, hist[OURS] = train_ours(env, cfg_k, s, **eval_kw)
    times[OURS] = time.perf_counter() - t0

    torch.manual_seed(s)
    t0 = time.perf_counter()
    qnet, hist["BDQ"] = train_bdq(env, cfg_k, seed=s, verbose=False, **eval_kw)
    times["BDQ"] = time.perf_counter() - t0

    torch.manual_seed(s)
    t0 = time.perf_counter()
    pnet, hist["TRPO"] = train_trpo(env, cfg_k, seed=s, verbose=False, **eval_kw)
    times["TRPO"] = time.perf_counter() - t0

    rng = lambda: torch.Generator().manual_seed(s + 1)
    agents = {
        "null": NullAgent(),
        "greedy": GreedyAgent(env.victim, env.theta_target, cfg_k, mode="greedy"),
        "BDQ": BDQAgent(qnet, env, cfg_k, rng()),
        "TRPO": TRPOAgent(pnet, env, cfg_k, rng(), deterministic=True),
        OURS: MethodAgent(V, env.victim, env.theta_target, fevi_cfg(cfg_k, env.T), rng()),
    }
    costs = {n: evaluate(env, a, torch.Generator().manual_seed(EVAL_SEED), EVAL_EPISODES)
             for n, a in agents.items()}
    return {"env": env, "agents": agents, "costs": costs, "hist": hist, "times": times}


runs = []
for s in SEEDS:
    t0 = time.perf_counter()
    runs.append(run_seed(s, K_COMPARE))
    c, tm = runs[-1]["costs"], runs[-1]["times"]
    print(f"seed {s}: " + "  ".join(f"{n}={c[n]:.3f} ({tm[n]:.0f}s)" for n in LEARNERS)
          + f"  greedy={c['greedy']:.3f}  null={c['null']:.3f}  [{time.perf_counter()-t0:.0f}s]")

## Results

### Episode cost

Error bars are ±1 std over seeds.

In [ ]:
cost_mean = {n: float(np.mean([r["costs"][n] for r in runs])) for n in NAMES}
cost_std = {n: float(np.std([r["costs"][n] for r in runs])) for n in NAMES}

fig, ax = plt.subplots(figsize=(8, 4.6))
ax.bar(NAMES, [cost_mean[n] for n in NAMES], yerr=[cost_std[n] for n in NAMES],
       capsize=5, color=[COLORS[n] for n in NAMES])
for i, n in enumerate(NAMES):
    ax.text(i, cost_mean[n] + 0.25, f"{cost_mean[n]:.2f}", ha="center")
ax.axhline(cost_mean["null"], color=COLORS["null"], ls="--", lw=1)
ax.set_ylabel("mean episode cost")
ax.set_title(f"d={ECFG.d}, m={ECFG.m}, T={ECFG.T}, c_edit={CFG.c_edit}, {len(SEEDS)} seeds")
plt.show()

print_raw(f"episode cost ({EVAL_EPISODES} eps)", {n: [r["costs"][n] for r in runs] for n in NAMES})

### Learning curves

Held-out cost after each training iteration, against iterations and against wall-clock
time. Shaded bands are ±1 std over seeds.

In [ ]:
def curve_stats(name):
    H = np.array([r["hist"][name] for r in runs])
    return H.mean(0), H.std(0)

it = np.arange(1, K_COMPARE + 1)
per_it = {n: np.mean([r["times"][n] for r in runs]) / K_COMPARE for n in LEARNERS}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for n in LEARNERS:
    mu, sd = curve_stats(n)
    for ax, x in [(a1, it), (a2, it * per_it[n])]:
        ax.plot(x, mu, f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n,
                markevery=max(1, K_COMPARE // 8))
        ax.fill_between(x, mu - sd, mu + sd, color=COLORS[n], alpha=0.15)
for ax in (a1, a2):
    ax.axhline(cost_mean["null"], color=COLORS["null"], ls="--", lw=1.5, label="null")
    ax.axhline(cost_mean["greedy"], color=COLORS["greedy"], ls="-.", lw=1.5, label="greedy")
a1.set_xlabel("training iteration")
a1.set_ylabel("held-out episode cost")
a2.set_xlabel("training time (s)")
a2.set_xscale("log")
a1.legend(loc="upper right", bbox_to_anchor=(0.98, 0.98))

for ax, tag in [(a1, "(a)"), (a2, "(b)")]:
    ax.text(0.5, -0.22, tag, transform=ax.transAxes, ha="center", va="top", fontsize=12)
plt.tight_layout()
plt.show()


print_raw("cost at last iteration", {n: [r["hist"][n][-1] for r in runs] for n in LEARNERS})
print_raw("cost at best iteration", {n: [min(r["hist"][n]) for r in runs] for n in LEARNERS})

### Compute

In [ ]:
def decision_latency_ms(env, agent, n=200, seed=1):
    rng = torch.Generator().manual_seed(seed)
    theta, z = env.theta_0.clone(), env.stream.sample_batch(env.m, rng)
    agent.act(theta, z, 0)                                   # warm-up
    t0 = time.perf_counter()
    for k in range(n):
        agent.act(theta, z, k % env.T)
    return (time.perf_counter() - t0) / n * 1e3

lat_raw = {n: [decision_latency_ms(r["env"], r["agents"][n]) for r in runs]
           for n in ["greedy", "TRPO", "BDQ", OURS]}
lat = {n: float(np.mean(v)) for n, v in lat_raw.items()}
train_t = {n: float(np.mean([r["times"][n] for r in runs])) for n in LEARNERS}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.bar(list(train_t), list(train_t.values()), color=[COLORS[n] for n in train_t])
a1.set_ylabel("seconds")
a1.set_title(f"Training time ({K_COMPARE} iterations)")
a2.bar(list(lat), list(lat.values()), color=[COLORS[n] for n in lat])
a2.set_ylabel("ms per action")
a2.set_title("Decision latency")
plt.tight_layout()
plt.show()

print_raw("training time (s)", {n: [r["times"][n] for r in runs] for n in LEARNERS}, 1)
print_raw("decision latency (ms)", lat_raw)

### Behaviour

Points edited per step, `Σ|aᵢ − 1|`, and distance to the target, `‖θ_t − θ̃‖`. The cell also
checks that every action is feasible.

In [ ]:
def behaviour(env, agent, n_episodes=100, seed=EVAL_SEED):
    rng = torch.Generator().manual_seed(seed)
    edits = gap = 0.0
    steps, feasible = 0, True
    for _ in range(n_episodes):
        theta = env.theta_0.clone()
        for t in range(env.T):
            z = env.stream.sample_batch(env.m, rng)
            a = agent.act(theta, z, t)
            feasible &= is_feasible(a, CFG.balance, CFG.B, CFG.L)
            edits += float(np.abs(a.numpy() - 1).sum())
            theta = env.victim.update(theta, z, a).detach()
            gap += float((theta - env.theta_target).norm())
            steps += 1
    return edits / steps, gap / steps, feasible

beh_raw = {n: np.array([behaviour(r["env"], r["agents"][n])[:2] for r in runs]) for n in NAMES}
beh = {n: v.mean(0) for n, v in beh_raw.items()}
feas = {n: all(behaviour(r["env"], r["agents"][n], 20)[2] for r in runs) for n in NAMES}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.bar(NAMES, [beh[n][0] for n in NAMES], color=[COLORS[n] for n in NAMES])
a1.set_ylabel("points edited per step")
a2.bar(NAMES, [beh[n][1] for n in NAMES], color=[COLORS[n] for n in NAMES])
a2.set_ylabel("‖θ_t − θ̃‖")
plt.tight_layout()
plt.show()

print("all actions feasible:", all(feas.values()), feas)
print_raw("edits per step", {n: beh_raw[n][:, 0] for n in NAMES})
print_raw("‖θ_t − θ̃‖", {n: beh_raw[n][:, 1] for n in NAMES})

### Training on a new configuration

Used by every section below.

In [ ]:
def build_agents(env, cfg, seed, learners):
    # train `learners` on env/cfg and return them with null and greedy
    ags = {"null": NullAgent(),
           "greedy": GreedyAgent(env.victim, env.theta_target,
                                 replace(cfg, select_mode="greedy"), mode="greedy")}
    rng = lambda: torch.Generator().manual_seed(seed + 1)
    kw = dict(best_checkpoint=True, eval_episodes=EVAL_EPISODES, eval_seed=EVAL_SEED)
    for name in learners:
        torch.manual_seed(seed)
        if name == "BDQ":
            ags[name] = BDQAgent(train_bdq(env, cfg, seed=seed, verbose=False, **kw),
                                 env, cfg, rng())
        elif name == "TRPO":
            ags[name] = TRPOAgent(train_trpo(env, cfg, seed=seed, verbose=False, **kw),
                                  env, cfg, rng(), deterministic=True)
        else:
            ags[name] = MethodAgent(train_ours(env, cfg, seed, **kw), env.victim,
                                    env.theta_target, fevi_cfg(cfg, env.T), rng())
    return ags

## Learned function (d = 1)

A separate 1-D run (D = 25 parameters), so the victim's whole function can be drawn. Every
policy sees the same data. Each panel shows one attack against the no-attack reference.

In [ ]:
RUN_FUNCTION = True

if RUN_FUNCTION:
    E1, C1 = replace(ECFG, d=1), replace(CFG, K_iters=K_COMPARE)
    torch.manual_seed(SEEDS[0])
    env1 = Environment(E1, C1)
    ags1 = build_agents(env1, C1, SEEDS[0], LEARNERS)

    rng = torch.Generator().manual_seed(EVAL_SEED)
    batches = [env1.stream.sample_batch(env1.m, rng) for _ in range(env1.T)]
    X = torch.cat([z[0].reshape(-1) for z in batches]).numpy()
    Y = torch.cat([z[1] for z in batches]).numpy()

    def run_on(agent):
        th, acts = env1.theta_0.clone(), []
        for t, z in enumerate(batches):
            a = agent.act(th, z, t)
            acts.append(a.detach())
            th = env1.victim.update(th, z, a).detach()
        return th, torch.cat(acts).numpy()

    grid = torch.linspace(float(X.min()), float(X.max()), 300)[:, None]
    f = lambda th: env1.victim.forward(th, grid).detach().numpy()
    xg = grid.reshape(-1).numpy()
    f_teacher = torch.sin(grid.reshape(-1) * env1.stream.theta_star).numpy()
    f_target = f(env1.theta_target)
    f_null = f(run_on(ags1["null"])[0])

    fig, axes = plt.subplots(2, 2, figsize=(11, 10), sharex=True, sharey=True)
    rows = []
    for ax, n, tag in zip(axes.flat, ["greedy", "TRPO", "BDQ", OURS], "abcd"):
        th, A = run_on(ags1[n])
        ax.scatter(X, Y, color="0.65", alpha=0.6, s=30)
        ax.plot(xg, f_teacher, color="0.3", ls=":", lw=1.8, label="teacher sin(θ*x)")
        ax.plot(xg, f_null, color="green", lw=2, label="null (no attack)")
        ax.plot(xg, f(th), color="blue", lw=2, label=f"{n} attack")
        ax.plot(xg, f_target, color="red", lw=2, label=r"target $f(\cdot;\tilde\theta)$")
        ax.scatter(X[A == 0], Y[A == 0], marker="x", color="red", s=45, label="deleted")
        ax.scatter(X[A == 2], Y[A == 2], marker="+", color="green", s=80, label="duplicated")
        ax.set_xlabel(r"$x$")
        ax.set_ylabel(r"$y$")
        ax.legend(loc="upper left", frameon=True, fontsize=8)
        ax.text(0.5, -0.14, f"({tag}) {n}", transform=ax.transAxes, ha="center", va="top",
                fontsize=12)
        rows.append((n, float((th - env1.theta_target).norm()),
                     float(np.sqrt(((f(th) - f_target) ** 2).mean())),
                     int((A == 0).sum()), int((A == 2).sum())))
    plt.tight_layout()
    plt.show()

    print(f"{'policy':8}{'‖θ_T − θ̃‖':>12}{'RMSE to target f':>18}{'deleted':>9}{'duplicated':>12}")
    for n, gap, rmse, d0, d2 in rows:
        print(f"{n:8}{gap:12.3f}{rmse:18.3f}{d0:9d}{d2:12d}")

## Scaling in `m`

The action space grows as `3^m`. Per-point influence shrinks as `1/m`, so `c_edit·m` is
held constant (`SCALE_CEDIT`) to keep edits equally affordable. TRPO is not trained here.

In [ ]:
RUN_SCALING = False
SCALE_CEDIT = True
MS = [4, 8, 12, 16]

if RUN_SCALING:
    SC_LEARNERS = [OURS, "BDQ"]
    SC_NAMES = ["null", "greedy"] + SC_LEARNERS
    sc = {n: [] for n in SC_NAMES}
    sc_t = {n: [] for n in SC_LEARNERS}
    for m in MS:
        cfg_m = replace(CFG, K_iters=20, c_edit=CFG.c_edit * ECFG.m / m if SCALE_CEDIT
                        else CFG.c_edit)
        torch.manual_seed(0)
        env_m = Environment(replace(ECFG, m=m), cfg_m)
        ags = build_agents(env_m, cfg_m, 0, [])
        for n in SC_LEARNERS:
            t0 = time.perf_counter()
            ags.update(build_agents(env_m, cfg_m, 0, [n]))
            sc_t[n].append(time.perf_counter() - t0)
        for n in SC_NAMES:
            sc[n].append(evaluate(env_m, ags[n], torch.Generator().manual_seed(EVAL_SEED),
                                  EVAL_EPISODES))
        print(f"m={m}: " + "  ".join(f"{n}={sc[n][-1]:.3f}" for n in SC_NAMES))

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2))
    for n in SC_NAMES:
        a1.plot(MS, np.array(sc[n]) / np.array(sc["null"]), f"{MARKERS[n]}-",
                color=COLORS[n], lw=2, label=n)
    for n in SC_LEARNERS:
        a2.plot(MS, sc_t[n], f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n)
    a1.set_ylabel("cost / null")
    a2.set_ylabel("training time (s)")
    a2.set_yscale("log")
    for ax in (a1, a2):
        ax.set_xlabel("batch size m")
        ax.set_xticks(MS)
        ax.legend()
    plt.tight_layout()
    plt.show()

## Sensitivity

One-dimensional sweeps around the default configuration. Every learner is retrained at
every point. Run the sections above first.

In [ ]:
RUN_CEDIT = False
RUN_M_SENS = False
RUN_T_SENS = False
RUN_VIOLATION = False

SWEEP_LEARNERS = [OURS, "BDQ", "TRPO"]


def sweep(values, make_cfgs, seeds=(0,), label="x", post_env=None, n_beh=40):
    # make_cfgs(v) -> (EnvConfig, MethodConfig); post_env(env, v) may modify the env
    names = ["null", "greedy"] + SWEEP_LEARNERS
    res = {n: {"cost": [], "cost_sd": [], "edits": []} for n in names}
    for v in values:
        ecfg_v, cfg_v = make_cfgs(v)
        costs = {n: [] for n in names}
        edits = {n: [] for n in names}
        for s in seeds:
            torch.manual_seed(s)
            env_v = Environment(ecfg_v, cfg_v)
            if post_env is not None:
                post_env(env_v, v)
            ags = build_agents(env_v, cfg_v, s, SWEEP_LEARNERS)
            for n in names:
                costs[n].append(evaluate(env_v, ags[n], torch.Generator().manual_seed(EVAL_SEED),
                                         EVAL_EPISODES))
                e_, _, ok = behaviour(env_v, ags[n], n_episodes=n_beh)
                assert ok, f"{n} took an infeasible action at {label}={v}"
                edits[n].append(e_)
        for n in names:
            res[n]["cost"].append(float(np.mean(costs[n])))
            res[n]["cost_sd"].append(float(np.std(costs[n])))
            res[n]["edits"].append(float(np.mean(edits[n])))
        sd = (lambda n: f"±{res[n]['cost_sd'][-1]:.3f}") if len(seeds) > 1 else (lambda n: "")
        print(f"{label}={v}: " + "  ".join(f"{n}={res[n]['cost'][-1]:.3f}{sd(n)}"
                                          for n in names), flush=True)
    return res


def plot_sweep(xs, res, xlabel, right="edits", xticks=None):
    # left: cost / null; right: edits per step, or FEVI's improvement over greedy
    nul = np.array(res["null"]["cost"])
    fig, (p1, p2) = plt.subplots(1, 2, figsize=(12, 4.2))
    for n in res:
        p1.plot(xs, np.array(res[n]["cost"]) / nul, f"{MARKERS[n]}-", color=COLORS[n],
                lw=2, label=n)
    p1.set_ylabel("cost / null")
    p1.legend()
    if right == "premium":
        g = np.array(res["greedy"]["cost"])
        p2.plot(xs, (g - np.array(res[OURS]["cost"])) / g * 100, f"{MARKERS[OURS]}-",
                color=COLORS[OURS], lw=2)
        p2.axhline(0, color="k", lw=0.8)
        p2.set_ylabel(f"{OURS} improvement over greedy (%)")
    else:
        for n in res:
            if n != "null":
                p2.plot(xs, res[n]["edits"], f"{MARKERS[n]}-", color=COLORS[n], lw=2, label=n)
        p2.set_ylabel("points edited per step")
    for p in (p1, p2):
        p.set_xlabel(xlabel)
        if xticks is not None:
            p.set_xticks(xticks)
    plt.tight_layout()
    plt.show()

### Edit price `c_edit`

In [ ]:
CEDITS = [0.0, 0.02, 0.05, 0.1, 0.5]

if RUN_CEDIT:
    res_ce = sweep(CEDITS, lambda v: (ECFG, replace(CFG, c_edit=v, K_iters=K_COMPARE)),
                   seeds=SEEDS, label="c_edit")
    plot_sweep(CEDITS, res_ce, "c_edit", xticks=CEDITS)

### Batch size `m`

Left: `c_edit` fixed. Right: `c_edit·m` fixed, which removes the effect of each edit
becoming less valuable as `m` grows.

In [ ]:
MS_SENS = [4, 6, 8, 10, 14]

if RUN_M_SENS:
    res_m = {}
    for scaled in (False, True):
        print("c_edit*m fixed" if scaled else "c_edit fixed")
        res_m[scaled] = sweep(
            MS_SENS,
            lambda m, sc=scaled: (replace(ECFG, m=m), replace(
                CFG, K_iters=K_COMPARE, c_edit=CFG.c_edit * ECFG.m / m if sc else CFG.c_edit)),
            seeds=(SEEDS[0],), label="m")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, scaled, title in [(axes[0], False, f"c_edit = {CFG.c_edit}"),
                              (axes[1], True, f"c_edit·m = {CFG.c_edit * ECFG.m:.1f}")]:
        r = res_m[scaled]
        nul = np.array(r["null"]["cost"])
        for n in r:
            ax.plot(MS_SENS, np.array(r[n]["cost"]) / nul, f"{MARKERS[n]}-",
                    color=COLORS[n], lw=2, label=n)
        ax.set_xlabel("batch size m")
        ax.set_xticks(MS_SENS)
        ax.set_title(title)
    axes[0].set_ylabel("cost / null")
    axes[0].legend()
    plt.tight_layout()
    plt.show()

### Horizon `T`

Greedy is FEVI with a zero value function, so the right panel shows what the learned value
function adds. At `T = 1` the two must coincide; the cell checks this.

In [ ]:
TS_SENS = [1, 2, 4, 6, 10, 14]

if RUN_T_SENS:
    res_T = sweep(TS_SENS, lambda T: (replace(ECFG, T=T), replace(CFG, K_iters=K_COMPARE)),
                  seeds=(SEEDS[0],), label="T")
    plot_sweep(TS_SENS, res_T, "horizon T", right="premium", xticks=TS_SENS)

    g1, f1 = res_T["greedy"]["cost"][0], res_T[OURS]["cost"][0]
    assert abs(g1 - f1) < 1e-6 * max(1.0, abs(g1)), f"T=1: greedy={g1}, FEVI={f1}"

### Violating action-linearity

FEVI assumes the victim's update is linear in the weights, `θ_{t+1} = θ_t + Σ aᵢ φᵢ`. Here the
true update is either norm-clipped (`τ = ρ ×` typical step norm) or takes `n` SGD steps, while
FEVI keeps the linear model. The table reports the measured violation
`‖true − linear‖ / ‖linear step‖` at each setting.

In [ ]:
class ClipVictim:
    # clips the aggregate update to norm tau; phi_batch stays linear
    def __init__(self, base, tau):
        self.base, self.tau = base, tau
        self.eta, self.task = base.eta, getattr(base, "task", "regression")

    def residual(self, theta, z):
        return self.base.residual(theta, z)

    def phi_batch(self, theta, z):
        return self.base.phi_batch(theta, z)

    def update(self, theta, z, a):
        step = a @ self.base.phi_batch(theta, z)
        nrm = step.norm()
        if self.tau is not None and float(nrm) > self.tau:
            step = step * (self.tau / nrm)
        return theta + step


class MultiStepVictim:
    # n SGD steps on the same reweighted batch; phi_batch still models one step
    def __init__(self, base, n_steps):
        self.base, self.n_steps = base, n_steps
        self.eta, self.task = base.eta, getattr(base, "task", "regression")

    def residual(self, theta, z):
        return self.base.residual(theta, z)

    def phi_batch(self, theta, z):
        return self.base.phi_batch(theta, z)

    def update(self, theta, z, a):
        th = theta
        for _ in range(self.n_steps):
            th = th + a @ self.base.phi_batch(th, z)
        return th


def linearity_error(env, agent, n_states=60, seed=EVAL_SEED):
    rng = torch.Generator().manual_seed(seed)
    errs, theta = [], env.theta_0.clone()
    for i in range(n_states):
        if i % env.T == 0:
            theta = env.theta_0.clone()
        z = env.stream.sample_batch(env.m, rng)
        a = agent.act(theta, z, i % env.T)
        pred = theta + a @ env.victim.phi_batch(theta, z)
        true = env.victim.update(theta, z, a).detach()
        errs.append(float((true - pred).norm() / ((pred - theta).norm() + 1e-12)))
        theta = true
    return float(np.mean(errs))


RHOS = [None, 1.0, 0.6, 0.4, 0.25, 0.12]      # None = no clipping
NSTEPS = [1, 2, 3, 4]

if RUN_VIOLATION:
    # typical step norm under the unclipped greedy policy
    torch.manual_seed(SEEDS[0])
    env0 = Environment(ECFG, replace(CFG, K_iters=K_COMPARE))
    g0 = GreedyAgent(env0.victim, env0.theta_target, CFG, mode="greedy")
    rng, norms, th = torch.Generator().manual_seed(EVAL_SEED), [], env0.theta_0.clone()
    for i in range(200):
        if i % env0.T == 0:
            th = env0.theta_0.clone()
        z = env0.stream.sample_batch(env0.m, rng)
        a = g0.act(th, z, i % env0.T)
        norms.append(float((a @ env0.victim.phi_batch(th, z)).norm()))
        th = env0.victim.update(th, z, a).detach()
    STEP0 = float(np.median(norms))

    clip = lambda vic, r: ClipVictim(vic, None if r is None else r * STEP0)
    res_clip = sweep(RHOS, lambda r: (ECFG, replace(CFG, K_iters=K_COMPARE)),
                     seeds=(SEEDS[0],), label="rho",
                     post_env=lambda env, r: setattr(env, "victim", clip(env.victim, r)))
    res_step = sweep(NSTEPS, lambda n: (ECFG, replace(CFG, K_iters=K_COMPARE)),
                     seeds=(SEEDS[0],), label="n_steps",
                     post_env=lambda env, n: setattr(env, "victim", MultiStepVictim(env.victim, n)))

    def violation(values, wrap):
        out = []
        for v in values:
            torch.manual_seed(SEEDS[0])
            e = Environment(ECFG, replace(CFG, K_iters=K_COMPARE))
            e.victim = wrap(e.victim, v)
            out.append(100 * linearity_error(
                e, GreedyAgent(e.victim, e.theta_target, CFG, mode="greedy")))
        return out

    vio_clip, vio_step = violation(RHOS, clip), violation(NSTEPS, MultiStepVictim)

    fig, (c1, c2) = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
    for ax, xs, res_, xlabel in [(c1, range(len(RHOS)), res_clip, "clip ρ"),
                                 (c2, NSTEPS, res_step, "SGD steps per round")]:
        nul = np.array(res_["null"]["cost"])
        for n in res_:
            ax.plot(list(xs), np.array(res_[n]["cost"]) / nul, f"{MARKERS[n]}-",
                    color=COLORS[n], lw=2, label=n)
        ax.set_xlabel(xlabel)
    c1.set_xticks(range(len(RHOS)))
    c1.set_xticklabels(["none" if r is None else f"{r:g}" for r in RHOS])
    c1.set_ylabel("cost / null")
    c1.legend()
    plt.tight_layout()
    plt.show()

    print(f"\n{'setting':>16}{'violation':>11}" + "".join(f"{n:>9}" for n in SWEEP_LEARNERS))
    for lbl, res_, vio, vals in [("rho", res_clip, vio_clip, RHOS),
                                 ("n_steps", res_step, vio_step, NSTEPS)]:
        nul = np.array(res_["null"]["cost"])
        for i, v in enumerate(vals):
            print(f"{lbl + '=' + ('none' if v is None else str(v)):>16}{vio[i]:>10.1f}%"
                  + "".join(f"{res_[n]['cost'][i] / nul[i]:>9.3f}" for n in SWEEP_LEARNERS))

## Summary

In [ ]:
print(f"d={ECFG.d} m={ECFG.m} T={ECFG.T} c_edit={CFG.c_edit}  seeds={list(SEEDS)}  "
      f"K={K_COMPARE}  eval episodes={EVAL_EPISODES}\n")
hdr = (f"{'policy':10}{'cost':>16}{'vs null':>9}{'edits/step':>12}{'‖θ−θ̃‖':>9}"
       f"{'train':>9}{'ms/action':>11}")
print(hdr)
print("-" * len(hdr))
for n in NAMES:
    red = (cost_mean["null"] - cost_mean[n]) / cost_mean["null"] * 100
    tt = f"{train_t[n]:.0f}s" if n in train_t else "-"
    ll = f"{lat[n]:.3f}" if n in lat else "-"
    print(f"{n:10}{cost_mean[n]:>9.3f} ±{cost_std[n]:<5.2f}{red:>8.1f}%"
          f"{beh[n][0]:>12.2f}{beh[n][1]:>9.2f}{tt:>9}{ll:>11}")